# 第 32 节：SmartReview 后端审查引擎实现 —— 把“会审条款的模型”，变成“能审合同的引擎”

> 前置课程：第 28~31 节（微调原理 / 指令数据集 / LoRA 训练 / 评估与量化）；模块四 RAG 检索链路
> （本节只“复用”检索结果，**不重复讲实现**）
> 前置知识：Python / SQLite / 正则基础；读过 `smartreview/backend` 的工程骨架
> 对应项目：SmartReview（`smartreview/`）—— 本节的代码即 `backend/` 审查引擎本体
> 对应大纲：`课程大纲.md —— 第 32 节`
> 运行环境：Python 3.10+；建议先执行 `pip install -r smartreview/backend/requirements.txt`

---

## 课程导航

| 部分 | 内容 | 对应文件（`smartreview/backend/`） |
|------|------|----------|
| **Part 1** | 课程导入：为什么 5000 字合同不能整份丢给模型 | — |
| **Part 2** | 环境准备 + “双通道 + 资料员”引擎全景 | `config/settings.py` |
| **Part 3** | 合同拆解器：正则“切条”为什么比模型靠谱 | `review/contract_splitter.py`、`review/metadata_extractor.py` |
| **Part 4** | 规则引擎：确定性通道（企业红线） | `rules.py`（规则数据在 DB `risk_rules`） |
| **Part 5** | RAG 知识供给（复用模块四，一笔带过） | `rag/*`（模块四成果） |
| **Part 6** | LLM 通道：四段式 Prompt 与裁决 JSON 容错解析 | `review/context_builder.py`、`review/verdict_parser.py` |
| **Part 7** | 并发调度与降级哲学 | `review/scheduler.py` |
| **Part 8** | 端到端引擎演示：规则 + LLM + 兜底三通道合体 | `review/scheduler.py` + `db/*` |
| **Part 9** | Word 审查报告生成器 | `review/report_generator.py` |
| **Part 10** | 演进回顾 + 课后挑战（收官预告第 33 节） | — |

---

> **面向学员**：延续第 24~27 节的 SmartBrief 教学思路，这份教程**不是背代码**，
> 而是带你重走真实项目的思考路径：**原始问题 → 粗暴解法 → 遇到痛点 → 重构优化 → 项目里的最终代码**。
> 代码里每一处“看着啰嗦 / 反直觉”的设计，背后都是一次线上教训。
> 建议按顺序逐 Cell 运行，观察每一次输出。


---
## Part 1：课程导入 —— 为什么 5000 字合同不能整份丢给模型
---

### 1.1 法务的诉求

第 31 节我们训出了审查模型，并量化成 8G 能跑的 vLLM 服务。此时产品经理把“法务老张”带了过来：

> “你们的模型很聪明，但法务不可能把一份 48 页的合同复制粘贴进对话框，
> 然后一条条问‘这条有没有风险’。能不能**传一个文件 → 自动把风险条款标出来 → 给一份 Word 报告**？”

你点点头，但心里清楚这里藏了三个**模型回答不了、必须靠工程解决**的问题：

| 问题 | 为什么丢给模型不行 |
|------|--------------------|
| **① 太长了** | 一份合同几千上万 token，塞进上下文会超限；即便塞得下，模型对中后部条款的“注意力”会明显衰减，漏判风险 |
| **② 会“一本正经胡说”** | 让模型自由判断，它可能引用一条不存在的法条；而“违约金上限 30%”这种**企业红线**必须 100% 命中、且能解释给法务听 |
| **③ 太慢** | 一次调用 1~2 秒，几十条条款串行要一分钟以上；任何一条请求失败还会让整单审查作废 |

### 1.2 工程解法：把“审合同”拆成流水线

真实风控团队审合同的分工是：**先有人把合同按“第 X 条”拆开 → 给每个条款查对应法条 → 红线规则先过一遍 → 吃不准的再请专家（大模型）判断 → 汇总成报告**。

SmartReview 把这套流程工程化，就是本节的引擎：

```
拆解线：整份合同文本  →(contract_splitter)→ 一条一条的"条款单元"
通道线：每一条 先走 确定性规则(红线) ─命中→ 直接裁决
                     未命中 → RAG 查法条 → 大模型审查 → JSON 裁决
兜底线：模型不可用/输出坏 → fallback("低风险·人工复核")，绝不拖垮整单
落库线：任务/结果/进度 → SQLite（review_tasks / review_results / risk_rules）
产出线：全部完成 → Word 审查报告
```

> ⚠️ 本节口径：`rag/`（检索）在模块四已讲透，这里**只说明它怎么被调用**，
> 把课堂时间留给本节真正的工程核心——拆解器、规则引擎、并发调度与容错。

### 1.3 本节课的“开发旅程”

| 步骤 | 我们解决什么 | 项目里的实现 |
|------|-------------|-------------|
| ① | 长合同没法整份喂模型 | 正则按“第 X 条”切条款（Part 3） |
| ② | 幻觉不可信、红线必须 100% 命中 | 确定性规则通道 + 双通道仲裁（Part 4） |
| ③ | 模型判断要“有据可依” | 检索公开法条拼进 Prompt（Part 5，复用） |
| ④ | LLM 输出不稳定 | 四段式 Prompt + JSON 容错解析 + 兜底（Part 6） |
| ⑤ | 串行太慢、单条失败拖垮整单 | 线程池并发 + 逐条异常隔离（Part 7） |
| ⑥ | 链路只存在于脑海 | 用假模型跑通三通道端到端（Part 8）+ 出报告（Part 9） |


---
## Part 2：环境准备与引擎全景
---

### 2.1 引擎的文件地图（先记住，别急着读代码）

| 文件 | 一句话职责 | 本节角色 |
|------|-----------|---------|
| `review/contract_splitter.py` | PDF/Word/TXT → 全文 + 按“第 X 条”拆条款 | ★ 拆解器 |
| `review/metadata_extractor.py` | 正则捞合同名称/甲方/乙方/总金额 | ★ 元数据 |
| `rules.py` | 三类确定性规则匹配器（纯函数） | ★ 规则引擎 |
| `review/context_builder.py` | 把公共信息+规则+条款+法条拼成 Prompt | LLM 通道 |
| `review/verdict_parser.py` | 把 LLM 的“自由文本”容错解析成裁决 JSON | LLM 通道 |
| `review/scheduler.py` | 并发调度逐条审查、进度落库 | ★ 调度器 |
| `review/report_generator.py` | 结果汇总成带颜色的 Word 报告 | 报告 |
| `db/database.py` + `db/review_repository.py` | 建表/seed + 任务与结果 CRUD | 落库 |
| `rag/*` | 法条加载/分块/向量化/检索（模块四） | 复用 |
| `config/settings.py` | 全局配置（路径/并发数/模型端点…） | 配置 |

> 阅读线索：本节按 **数据流** 走读——`拆解 → 判断（规则/LLM）→ 调度 → 报告`，
> 和代码目录顺序一致。`config/` 是全局配置，任何模块都不自己读环境变量。

### 2.2 “双通道 + 资料员”：本系统最核心的架构决定

```
         ┌─────────────────────────── 一条条款 ───────────────────────────┐
         │                                                                │
  (资料员) RAG 法条检索 ──┐                                                  │
                         ▼                                                │
          ┌──── 通道一：确定性规则(企业红线) ──命中──▶ 直接裁决(channel=rule)    │
          │ 未命中                                                          │
          ▼                                                                │
      通道二：大模型(LLM) 审查  ──▶ JSON 容错解析 ──▶ 裁决(channel=llm)         │
          │        重试≤2 仍失败 / 模型不可用                                 │
          ▼                                                                │
      fallback_verdict（低风险·人工复核，channel=fallback）                   │
         └──────────────────────────────────────────────────────────────┘
```

一句话记忆：**规则通道管“红线”（毫秒级、零幻觉、可解释）；LLM 通道管“疑难”（能泛化、慢）；RAG 只当资料员（喂法条、压幻觉），不当裁判**。

### 2.3 先跑环境检查

下面这个 Cell 把 `backend/` 加进 `sys.path`，并做一件重要的事：**把配置指向一次性临时沙箱**。
这样本 notebook 的演示（建库、写文件）都不会碰项目真实 `data/` 目录与 `.env` 指向的库——教学演示可以随意反复跑。


In [1]:
# ============================================================
# Cell 1：环境检查 + sys.path + 演示专用临时沙箱
# ============================================================
import os
import sys
import tempfile
from pathlib import Path

print("=" * 62)
print("环境检查（第 32 节）")
print("=" * 62)
print(f"  Python 版本: {sys.version.split()[0]}")

# 兼容两种运行位置：
#   1) notebook 在 smartreview/ 目录下运行 -> backend = ./backend
#   2) notebook 被拷到 06-finetune/ 根目录运行 -> backend = ./smartreview/backend
BACKEND = Path(os.getcwd()) / "backend"
if not BACKEND.is_dir():
    BACKEND = Path(os.getcwd()) / "smartreview" / "backend"
BACKEND = BACKEND.resolve()
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))
SMARTREVIEW = BACKEND.parent
print(f"  backend 路径 : {BACKEND}")
print(f"  smartreview : {SMARTREVIEW}")

# 关键：在导入任何"会建库/碰向量库"的模块之前，把配置指到一次性临时目录。
# 演示绝不读写真实 data/、也绝不使用 .env 里的真实业务库。
from config.settings import settings  # noqa: E402

SANDBOX = Path(tempfile.mkdtemp(prefix="sm32_"))
settings.upload_dir = str(SANDBOX / "uploads")
settings.report_dir = str(SANDBOX / "reports")
settings.kb_dir = str(SANDBOX / "kb")
settings.chroma_persist_dir = str(SANDBOX / "chroma")
settings.db_path = str(SANDBOX / "demo.db")
settings.ensure_dirs()
print(f"  演示沙箱目录: {SANDBOX}")

# 核心模块可导入性检查
for mod in ("review.contract_splitter", "review.metadata_extractor", "rules"):
    try:
        __import__(mod, fromlist=["*"])
        print(f"  [OK] {mod} 可导入")
    except Exception as exc:
        print(f"  [警告] {mod}: {exc}（请先 pip install -r backend/requirements.txt）")


环境检查（第 32 节）
  Python 版本: 3.10.20
  backend 路径 : C:\Users\13691\Documents\Code\260601-GPKJ\06-finetune\smartreview\backend
  smartreview : C:\Users\13691\Documents\Code\260601-GPKJ\06-finetune\smartreview
  演示沙箱目录: C:\Users\13691\AppData\Local\Temp\sm32_6mp1drva
  [OK] review.contract_splitter 可导入
  [OK] review.metadata_extractor 可导入
  [OK] rules 可导入


---
## Part 3：合同拆解器 —— 正则“切条”为什么比模型靠谱
---

### 3.1 核心问题

模型能审“一条”条款，但合同是“一整份”。**第一件事：把整份合同切成一条条独立的“审查单元”。**

粗暴解 v0 有两个选项，都很快翻车：

| 粗暴方案 | 痛点 |
|---------|------|
| 整份合同一次喂模型 | 超长截断；中后部条款注意力衰减、漏判 |
| 按固定字数（如 500 字）硬切 | 切在句子中间；条号归属混乱；“第五条的正文”可能被切进“第六条” |

工程上真正合适的粒度，其实是**合同的天然结构单位——条款（第 X 条）**：
一条条款 = 一个独立的风险判断单元，内部“项 / 款”不用再拆（语义依赖父条款上下文）。

### 3.2 定位“第 X 条”：一个正则就能做到

先做个小实验——用正则把样例合同里的条号全部找出来（不读代码，先验证思路）：


In [2]:
# ============================================================
# Cell 2：条号正则小实验 —— 切分锚点长什么样
# ============================================================
import re

SAMPLE = SMARTREVIEW / "samples" / "contracts" / "软件委托开发合同样例.txt"
text = SAMPLE.read_text(encoding="utf-8")
print(f"样例合同：{SAMPLE.name}（{len(text)} 字符）\n")

# 中文 + 阿拉伯数字条款标记
CLAUSE_PAT = re.compile(r"第\s*([零一二三四五六七八九十百千万\d]+)\s*条")
hits = [(m.group(0), m.start()) for m in CLAUSE_PAT.finditer(text)]
print(f"找到 {len(hits)} 个条款标记（条号, 全文偏移）:")
for name, pos in hits:
    print(f"  {name:<4} char_index={pos}")


样例合同：软件委托开发合同样例.txt（778 字符）

找到 10 个条款标记（条号, 全文偏移）:
  第一条  char_index=77
  第二条  char_index=141
  第三条  char_index=202
  第四条  char_index=281
  第五条  char_index=357
  第六条  char_index=445
  第七条  char_index=498
  第八条  char_index=592
  第九条  char_index=661
  第十条  char_index=718


### 3.3 只找到“条号”还不够——拆解器的隐藏工程细节

有了条号这个“锚点”，拆条款看似简单（条号 A 的下一条号 B 之间就是 A 的正文）。
但直接上线的拆解器还得多想几步，**每一步都对应一个真实事故**：

| 细节 | 为什么必须处理 |
|------|----------------|
| ① 找不到“第 X 条”结构 | 采购单、补充协议可能整篇没有条号 → 必须降级：按空行段落硬切，`clause_id` 用 `P1/P2…`，**系统不报错** |
| ② 条号后的“标题” | 报告里要“第一条 违约责任”这样的定位标题 → 从条号后行首抽一句，去掉 `、.：` 等噪音 |
| ③ 条款在合同哪一页 | 法务要按页翻原文核对 → 记录每个字在全文的偏移，二分查 `_page_of` 定位页码 |
| ④ 页眉页脚噪音 | 分页文本里重复页眉会污染正文 → 拆条前按非空行切分、压缩换行 |
| ⑤ 条款正文可能为空 | 空条款直接跳过，不产生垃圾审查任务 |

> **为什么用正则而不是用模型切？** 稳定（同样输入永远同样输出）、毫秒级、零成本；
> 而且切分错了会让下游“审错对象”——确定性结构解析不该交给有随机性的模型。
> 这正是 `review/contract_splitter.py` 的取舍。

现在去读项目里的真实实现，看看这些细节最终长什么样：


In [3]:
# ============================================================
# Cell 3：走读真实实现 split_clauses + _page_of
# ============================================================
import inspect
from review.contract_splitter import _page_of, split_clauses

for fn in (split_clauses, _page_of):
    print("=" * 62)
    print(f"  {fn.__module__}.{fn.__name__}")
    print("=" * 62)
    print(inspect.getsource(fn))


  review.contract_splitter.split_clauses
def split_clauses(text: str, page_offsets: Optional[List[int]] = None) -> List[dict]:
    """
    正则切分条款。返回每条：clause_id / title / content / char_index / page。
    降级：无结构时按空行段落硬切，clause_id 使用 P1/P2…。
    """
    page_offsets = page_offsets or [0]

    matches = list(_CLAUSE_PATTERN.finditer(text))
    if len(matches) >= 2:
        clauses = []
        for idx, m in enumerate(matches):
            start = m.end()
            end = matches[idx + 1].start() if idx + 1 < len(matches) else len(text)
            content = text[start:end].strip()
            if not content:
                continue
            clauses.append({
                "clause_id": m.group(0).strip(),
                "title": _extract_title(text[start:end]),
                "content": content,
                "char_index": m.start(),
                "page": _page_of(m.start(), page_offsets),
            })
        return clauses

    # ---- 降级：按段落硬切 ----
    clauses = []
    para

### 3.4 跑真实拆解器 + 元数据提取

`parse_contract` 是拆解入口：读文件 → 拼全文并记录每页偏移 → 切条款。
`extract_metadata` 负责“合同头”的公共信息（名称 / 甲方 / 乙方 / 总金额），这些会被送进每一条的 Prompt 做背景。

注意 `extract_metadata` 的模块头写着：**提取结果仅在内存中传递，绝不写入日志**——
合同含敏感商业信息，这是 PRD 的硬性合规要求，不是小题大做。


In [4]:
# ============================================================
# Cell 4：跑真实 parse_contract + extract_metadata
# ============================================================
import json
from review.contract_splitter import parse_contract
from review.metadata_extractor import extract_metadata

parsed = parse_contract(str(SAMPLE))
clauses = parsed["clauses"]

print(f"page_count = {parsed['page_count']}（TXT 按行数预估）   条款数 = {len(clauses)}")
print("\n前 6 条预览（clause_id | title | 页 | 正文字数 | char_index）:")
for c in clauses[:6]:
    print(f"  {c['clause_id']:<5} {c['title']!r:<18} 页={c['page']}  len={len(c['content']):<4} char={c['char_index']}")

meta = extract_metadata(parsed["full_text"], SAMPLE.name)
print("\n元数据抽取（合同头）:")
for k, v in meta.items():
    print(f"  {k:<14} = {v!r}")

# 组装成下游要消费的 contract_structure.json，落到沙箱
contract_structure = {
    "metadata": meta,
    "page_count": parsed["page_count"],
    "clauses": clauses,
}
out = SANDBOX / "contract_structure.json"
out.write_text(json.dumps(contract_structure, ensure_ascii=False, indent=2),
               encoding="utf-8")
print(f"\n已写出 contract_structure.json（{out.stat().st_size} 字节）→ 审查引擎的输入产物")


page_count = 1（TXT 按行数预估）   条款数 = 10

前 6 条预览（clause_id | title | 页 | 正文字数 | char_index）:
  第一条   '项目内容与交付物'         页=1  len=59   char=77
  第二条   '开发周期'             页=1  len=56   char=141
  第三条   '合同费用与支付'          页=1  len=74   char=202
  第四条   '验收标准'             页=1  len=71   char=281
  第五条   '知识产权与成果归属'        页=1  len=83   char=357
  第六条   '保密条款'             页=1  len=48   char=445

元数据抽取（合同头）:
  contract_name  = '软件委托开发合同'
  party_a        = '极电科技有限公司'
  party_b        = '云帆信息技术有限公司'
  total_amount   = '300,000元'

已写出 contract_structure.json（3662 字节）→ 审查引擎的输入产物


### 3.5 输出解读：拆解器交出了什么

每条条款是一个独立 dict：`clause_id`（原文条号）、`title`（抽出的标题）、`content`（正文）、
`char_index`（在全文的字符偏移，用于原文回溯）、`page`（所在页）。

**至此，“整份合同”变成了 N 条互不依赖的“审查单元”**——互不依赖，是后面能并发审查的前提（Part 7）。

拆解完成后，判断权就交给两个“裁判”：先看确定性规则通道（Part 4）。


---
## Part 4：规则引擎 —— 为什么必须有一条“确定性通道”
---

### 4.1 双通道的由来：一次事故

假设只有大模型一个裁判，会出现这种对话：

> 法务：“这份合同违约金上限是合同总额的 30%，为什么不给我标红？”
> 开发：“模型认为……呃，它这次判断没问题。”
> 法务：“上个月它还说 30% 没问题呢，这个月怎么又标红了？”

**LLM 的每一次判断都带随机性**（采样温度）。但企业红线（违约金上限 ≤20%、知识产权必须甲方独占、管辖地必须在甲方所在地……）是**写了进 PRD 的硬指标**，
不可能依赖“这次抽签运气好”。于是架构变成两个裁判：

| | 通道一：确定性规则 | 通道二：大模型审查 |
|---|---|---|
| 判断依据 | 正则 + 数值/关键词匹配 | 语义理解 + 泛化 |
| 幻觉风险 | 无（只按规则文本判断） | 有（可能引用不存在的法条） |
| 可解释性 | 强：命中规则名直接可写进报告 | 弱：靠 risk_description 解释 |
| 速度 / 成本 | 毫秒级 / ≈0 | 1~2 秒 / 每次计费 |
| 适用 | 红线、模式明确 | 诱导性 / 模糊 / 复杂条款 |

> ⚠️ 注意规则引擎的文件位置：`backend/rules.py`（在 backend 根，**不在** review/ 包内）。
> 它是纯函数模块（不依赖 DB、不依赖网络），所以能独立单测——这也是它为什么好维护。

### 4.2 三类匹配器：把“法务语言”翻译成代码

| matcher_kind | 判定逻辑 | 典型规则 |
|---|---|---|
| `percent_ceiling` | 正则抓百分比数值，超过阈值即命中（数值上限类） | 违约金/赔偿上限超标 |
| `ip_rights` | 涉知识产权且**非甲方独占**时命中 | 知识产权归属不当 |
| `keyword_any` | 命中任一预设关键词 | 争议管辖地在乙方所在地 |

三套逻辑合并成 3 条“出厂规则”（`DEFAULT_RULES`，在 `db/review_repository.py` 顶部）：
`违约金/赔偿上限超标`（高）、`知识产权归属不当`（中）、`争议管辖地不利`（中）。
规则数据存 DB `risk_rules` 表，管理端可增删改——**默认规则只在建库时 seed 一次，删了不复活**（这个设计第 33 节演示）。

先读真实源码：


In [5]:
# ============================================================
# Cell 5：走读规则引擎的三个匹配器
# ============================================================
import inspect
from rules import build_rule_verdict, match_ip_rights, match_percent_ceiling

for fn in (match_percent_ceiling, match_ip_rights):
    print("=" * 62)
    print(f"  rules.{fn.__name__}")
    print("=" * 62)
    print(inspect.getsource(fn))

print("=" * 62)
print("  build_rule_verdict —— 命中规则 -> 与 LLM 同构的裁决")
print("=" * 62)
print(inspect.getsource(build_rule_verdict))


  rules.match_percent_ceiling
def match_percent_ceiling(content: str, params: dict) -> bool:
    """命中：条款中出现高于阈值的百分比数值（默认 20% 企业底线）。"""
    regex = params.get("regex") or r"(\d+(?:\.\d+)?)\s*%"
    threshold = float(params.get("threshold", 20.0))
    for m in re.finditer(regex, content):
        try:
            if float(m.group(1)) > threshold:
                return True
        except (ValueError, IndexError):
            continue
    return False

  rules.match_ip_rights
def match_ip_rights(content: str) -> bool:
    """命中：涉知识产权且约定非甲方独占（共有/归乙方等表述）。"""
    ip_hit = re.search(r"著作权|知识产权|专利权|商标权|技术成果|软件著作权", content)
    if not ip_hit:
        return False
    if re.search(r"甲方\s*(独占|独有|唯一|全部所有)|归甲方\s*(所有|独有|享有)", content):
        return False
    return bool(re.search(r"共有|共同所有|双方共同|归双方|归乙方|乙方所有|供方所有", content))

  build_rule_verdict —— 命中规则 -> 与 LLM 同构的裁决
def build_rule_verdict(rule: dict) -> dict:
    """把命中的规则转成与 LLM 裁决同构的 verdict。"""
    name = rule.get("name", "")
    return {


### 4.3 用真实规则跑一遍：哪些该命中、哪些不该

下面直接调用真实 `evaluate_clause_rules`，喂 7 段典型条款文本，看规则通道的判定
（判据：percent 高于 20% 红线 / 知识产权非甲方独占 / 关键词命中；多规则命中取严重度最高者）。


In [6]:
# ============================================================
# Cell 6：规则通道命中测试（真实 rules.py + 出厂规则）
# ============================================================
import json as _json
from db.review_repository import DEFAULT_RULES
from rules import evaluate_clause_rules


def default_rules():
    '''出厂规则(params_json) -> 引擎可消费形态（与 get_enabled_rules 同逻辑）。'''
    out = []
    for r in DEFAULT_RULES:
        d = dict(r)
        d["params"] = _json.loads(d.pop("params_json")) or {}
        out.append(d)
    return out


cases = [
    ("违约金上限 30%（超红线，应命中·高）",
     "乙方逾期交付的，违约金上限为合同总金额的30%，上不封顶。"),
    ("违约金上限 15%（未超红线，不应命中）",
     "违约金总额以不超过合同总金额的15%为限。"),
    ("知识产权归双方共有（应命中·中）",
     "开发过程中产生的软件著作权归双方共同所有。"),
    ("知识产权甲方独占（不应命中）",
     "项目开发产生的技术成果的知识产权均归甲方独占所有。"),
    ("管辖地约定在乙方所在地（应命中·中）",
     "协商不成的，任何一方均可向乙方所在地人民法院提起诉讼。"),
    ("管辖地约定在甲方所在地（不应命中）",
     "协商不成的，任何一方均可向甲方所在地人民法院提起诉讼。"),
    ("两条规则同时命中（高+中，应取高）",
     "违约金上限为合同总额的30%；争议由乙方所在地法院管辖。"),
]
for name, content in cases:
    v = evaluate_clause_rules(content, default_rules())
    print(f"  [{'命中' if v else '--'}] {name}")
    if v:
        print(f"        -> risk_level={v['risk_level']}  描述={v['risk_description'][:40]}...")


  [命中] 违约金上限 30%（超红线，应命中·高）
        -> risk_level=高  描述=【规则:违约金/赔偿上限超标】违约金/赔偿上限可能超过企业红线（合同总额的20%...
  [--] 违约金上限 15%（未超红线，不应命中）
  [命中] 知识产权归双方共有（应命中·中）
        -> risk_level=中  描述=【规则:知识产权归属不当】知识产权权属约定不利于甲方（非甲方独占）。...
  [--] 知识产权甲方独占（不应命中）
  [命中] 管辖地约定在乙方所在地（应命中·中）
        -> risk_level=中  描述=【规则:争议管辖地不利】争议管辖/仲裁地约定可能增加甲方应诉成本。...
  [--] 管辖地约定在甲方所在地（不应命中）
  [命中] 两条规则同时命中（高+中，应取高）
        -> risk_level=高  描述=【规则:违约金/赔偿上限超标】违约金/赔偿上限可能超过企业红线（合同总额的20%...


### 4.4 记住三件事

1. **规则命中 = 直接裁决**（channel=`rule`，置信度恒定 0.95），不再调用模型；
2. 命中时若 RAG 有检索结果，会把最相关法条原文附到 `law_basis`——**结论可追溯**；
3. 命中裁决里带 `rule_name`（写在 `risk_description` 的 `【规则:xxx】` 前缀），报告里法务一眼能看懂“是哪条红线拦住的”。

规则通道没拦住的条款，才轮到 LLM——但喂给 LLM 之前，先要让 RAG 资料员去查法条（Part 5）。


---
## Part 5：RAG 知识供给（复用模块四，一笔带过）
---

### 5.1 这里 RAG 的角色：资料员，不是裁判

模块四已经把“加载→分块→向量化→检索”讲透了，SmartReview 只是**调用方**。调度器里对每条条款做一次检索：

```python
query = f"{clause.get('clause_id')} {title} {content[:200]}".strip()
hits = kb.retrieve(query)          # 语义检索 Top-K（不可用时自动降级为空）
law_context = kb.format_context(hits)   # 拼成"【检索到的相关法律条文】"块
```

要点（为什么这样写）：

| 设计 | 原因 |
|------|------|
| 用“条号 + 标题 + 正文前 200 字”当查询 | 检索意图要短而聚焦；全文太长会稀释向量相似度 |
| 检索失败只降级为空、不抛错 | 知识库缺 Key / 没语料时，审查链路不能因此中断 |
| 只把检索结果“喂”给 LLM，不参与决策 | RAG 管“有什么法条可用”，规则/LLM 才管“怎么判” |

法律语料（`data/kb/law/*.txt`，如《民法典》违约金与损失赔偿、技术合同成果归属等）在模块四已入库；
本节不再展开向量库细节。下一节看这些法条是怎么拼进 Prompt、以及模型输出怎么被“驯服”成结构化 JSON。


---
## Part 6：LLM 通道 —— 四段式 Prompt 与裁决 JSON 的容错解析
---

### 6.1 上下文组装：给模型的“工单”为什么是四段式

`review/context_builder.py` 把一次审查要用的材料拼成 **system + user** 两条消息，
user 里按固定顺序放四个信息块：

| 块 | 内容 | 作用 |
|----|------|------|
| ① 检索到的法律条文 | RAG 结果 + “引用需忠实原文，材料外禁止引用” | 压幻觉：不许编法条 |
| ② 企业风控规则 | 启用的规则清单（红线语义） | 让模型知道企业底线，避免与规则通道结论打架 |
| ③ 合同公共信息 | 名称/甲方/乙方/金额 | 判断“谁违约对谁不利”的背景 |
| ④ 待审查条款 | 条号+标题+正文 | 审查对象本体 |
| + 输出约束 | `VERDICT_SCHEMA`：必须输出**裸 JSON**（无解释文字） | 让下游能机器解析 |

> 为什么固定这套结构？因为第 29 节微调时，训练样本就是按这个四段式构造的（对照 `config/prompts.py` 的 `REVIEW_SYSTEM_PROMPT`）——
> **训练时喂什么形状，推理时就给什么形状**，模型表现才最稳。

### 6.2 现实很骨感：LLM 输出永远是“自由文本”

就算 Prompt 写“必须输出裸 JSON”，模型仍可能：用 ```json 围栏包起来、前面加一句“好的，审查结果如下：”、
把等级写成“高风险”而不是枚举值“高”……这就是 `review/verdict_parser.py` 存在的原因：
**对自由文本做三道“消毒”：剥围栏 → 取首个 JSON 对象 → 字段校验/归一化**；失败抛 `ParseError` 交给上层重试或兜底。

先看真实组装结果：


In [7]:
# ============================================================
# Cell 7：跑真实 build_review_messages —— 看 Prompt 的形状
# ============================================================
from review.context_builder import build_review_messages

# 用 Part 3 拆出的真实条款（第七条·违约责任），配两条风控规则做演示
demo_rule = {"name": "违约金/赔偿上限超标", "risk_level": "高",
             "description": "违约金/赔偿上限可能超过企业红线（合同总额的20%）"}
law_context = "民法典第五百八十五条：约定的违约金过分高于造成的损失的，人民法院可以根据当事人的请求予以适当减少。"

msgs = build_review_messages(
    contract_meta={"contract_name": "软件委托开发合同", "party_a": "极电科技",
                   "party_b": "云帆信息", "total_amount": "300,000元"},
    clause=clauses[6],
    law_context=law_context,
    rules=[demo_rule],
)
print(f"messages 共 {len(msgs)} 条：roles = {[m['role'] for m in msgs]}")
print(f"\nsystem 提示词（前 140 字）:\n  {msgs[0]['content'][:140].replace(chr(10), ' ')}...")

user = msgs[1]["content"]
print(f"\nuser 内容总长度：{len(user)} 字符，包含的关键块：")
for marker in ("【检索到的相关法律条文】", "【企业风控规则】", "【合同公共信息】", "【待审查条款】"):
    print(f"  ✓ {marker}  {'已包含' if marker in user else '缺失!'}")
print(f"\nuser 末尾输出约束片段：\n  {user[-120:].replace(chr(10), ' ')}")


messages 共 2 条：roles = ['system', 'user']

system 提示词（前 140 字）:
  你是资深的企业法律合同审查专家。你的任务是**逐条**审查合同条款， 依据提供的企业风控规则与检索到的法律条文，输出结构化的风险裁决。  审查要求： 1. 只审查给定的这一条条款，不要臆测其他条款内容。 2. 风险等级只能是以下四者之一："高"、"中"、"低"、"无"。 3. 判...

user 内容总长度：526 字符，包含的关键块：
  ✓ 【检索到的相关法律条文】  已包含
  ✓ 【企业风控规则】  已包含
  ✓ 【合同公共信息】  已包含
  ✓ 【待审查条款】  已包含

user 末尾输出约束片段：
  述：条款内容 + 判断依据",   "law_basis": "法条依据原文，无引用时填空字符串",   "suggestion": "具体修改建议，无风险时写'无需修改'",   "confidence": 0.0 到 1.0 的数字 }


In [8]:
# ============================================================
# Cell 8：走读真实 verdict_parser —— 三道“消毒” + 兜底
# ============================================================
import inspect
from review.verdict_parser import extract_json_object, parse_verdict

for fn in (extract_json_object, parse_verdict):
    print("=" * 62)
    print(f"  verdict_parser.{fn.__name__}")
    print("=" * 62)
    print(inspect.getsource(fn))


  verdict_parser.extract_json_object
def extract_json_object(text: str) -> dict:
    """从任意文本中提取首个 JSON 对象。"""
    # 移除 Markdown 代码块围栏
    text = re.sub(r"```(?:json)?", "", text)
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ParseError("未找到 JSON 对象")
    try:
        return json.loads(text[start:end + 1])
    except json.JSONDecodeError as e:
        raise ParseError(f"JSON 解码失败: {e}")

  verdict_parser.parse_verdict
def parse_verdict(text: str) -> dict:
    """
    将 LLM 输出解析为规范化 verdict：
    {risk_level, risk_description, law_basis, suggestion, confidence}
    """
    raw = extract_json_object(text)
    risk_level = _coerce_risk_level(raw.get("risk_level"))
    try:
        confidence = max(0.0, min(1.0, float(raw.get("confidence", 0.5))))
    except (TypeError, ValueError):
        confidence = 0.5

    verdict = {
        "risk_level": risk_level,
        "risk_description": str(raw.get("risk_description") or "").st

In [9]:
# ============================================================
# Cell 9：LLM 自由文本 -> 裁决 JSON 的容错实战
# ============================================================
from review.verdict_parser import ParseError, fallback_verdict, parse_verdict

# 场景 A：模型老实给了 ```json 围栏包着的裸 JSON
ok1 = '''```json
{"risk_level": "高", "risk_description": "违约金过高", "law_basis": "民法典585条", "suggestion": "降至20%", "confidence": 0.9}
```'''
# 场景 B：模型忍不住加了开场白，还写了口语化的"高风险"
ok2 = '好的，审查结果如下：{"risk_level": "高风险", "risk_description": "违约金过高", "law_basis": "", "suggestion": "修改", "confidence": 0.85} 请查收'
# 场景 C：模型彻底摆烂
bad = "抱歉我无法完成审查。"

for label, s in zip(("A. 带围栏", "B. 口语等级+夹带文字", "C. 摆烂"), (ok1, ok2, bad)):
    try:
        v = parse_verdict(s)
        print(f"  场景{label} -> 解析成功 risk_level={v['risk_level']} conf={v['confidence']}")
    except ParseError as e:
        print(f"  场景{label} -> ParseError: {e}")

fb = fallback_verdict("模型服务不可用")
print("\n  兜底裁决 fallback_verdict:")
print(f"    risk_level={fb['risk_level']}  conf={fb['confidence']}")
print(f"    描述: {fb['risk_description']}")


  场景A. 带围栏 -> 解析成功 risk_level=高 conf=0.9
  场景B. 口语等级+夹带文字 -> 解析成功 risk_level=高 conf=0.85
  场景C. 摆烂 -> ParseError: 未找到 JSON 对象

  兜底裁决 fallback_verdict:
    risk_level=低  conf=0.0
    描述: 未能完成自动判断（模型服务不可用），建议人工复核该条款。


### 6.3 兜底裁决的“语义”非常讲究

```python
fallback_verdict("模型服务不可用")
# {"risk_level": "低", "risk_description": "未能完成自动判断（模型服务不可用），建议人工复核该条款。", ...}
```

把“没审成”标成 **低风险** 而不是报错，是刻意设计：

1. **不进高风险统计**：汇总时不会把“没审”误算成“高风险项”；
2. **置信度 0 + 明示人工复核**：法务看到 conf=0 就知道这不是机器结论，必须人看；
3. **不拖垮整单**：30 条里 1 条模型超时，其余 29 条照常出报告——总比整单失败强。

接下来，把这些单条款能力交给调度器，用“并发”把 60 秒压到 8 秒。


---
## Part 7：并发调度与降级哲学
---

### 7.1 为什么能并发？—— 拆解器早就埋好了伏笔

Part 3 拆出的每条条款**相互独立、没有依赖**，天然适合并行。审查的本质是“等模型网络响应”（IO 密集），
所以选择 **线程池**（`ThreadPoolExecutor(8)`）而非多进程：

| 方案 | 为什么不用 |
|------|-----------|
| 多进程 | 审查函数在内存里，无共享大对象；多进程要序列化、更重 |
| asyncio | 底层 LLM 客户端/检索都是同步阻塞库，改造收益小、侵入大 |
| **线程池** | 8 个线程等网络 IO，几乎不占 CPU；代码改动最小（`executor.submit` + `as_completed`） |

先做个“同构但缩小版”的实验，直观感受串行 vs 并发（8 个任务，每个“审”0.05~0.15 秒）：


In [10]:
# ============================================================
# Cell 10：串行 vs 线程池 —— 完成顺序与耗时对比
# ============================================================
import concurrent.futures as cf
import time


def fake_review(i, t):
    time.sleep(t)          # 模拟一次 LLM 网络等待
    return i               # 返回"第几条审查完成"


N = 8
delays = [0.05 + i * 0.02 for i in range(N)]

def bench(label, workers):
    t0 = time.perf_counter()
    if workers == 1:
        order = [fake_review(i, delays[i - 1]) for i in range(1, N + 1)]
    else:
        with cf.ThreadPoolExecutor(max_workers=workers) as ex:
            futs = {ex.submit(fake_review, i, delays[i - 1]): i for i in range(1, N + 1)}
            order = [futs[f] for f in cf.as_completed(futs)]  # 完成顺序，不是提交顺序
    print(f"  {label:<18} 总耗时 {time.perf_counter() - t0:.3f}s")
    print(f"                    完成顺序 = {order}")

bench("串行(1线程)", 1)
bench("并发(8线程)", 8)


  串行(1线程)            总耗时 1.016s
                    完成顺序 = [1, 2, 3, 4, 5, 6, 7, 8]
  并发(8线程)            总耗时 0.205s
                    完成顺序 = [1, 2, 3, 4, 5, 6, 7, 8]


> 上面“完成顺序”是乱的——这正是 `as_completed` 的语义：**谁先完成谁先落库**。
> 真实项目里 30 条合同、每条约 2 秒：串行约 60s，8 线程并发只要 **7~8s**。
> 结果乱序落库没关系，因为每条结果都带 `clause_order` 列，报告按它还原顺序即可。

### 7.2 逐条审查函数：双通道逻辑的“心脏”

调度器的核心是 `_review_one_clause`：检索法条 → 规则通道优先 → LLM 通道 → 兜底。
读一下真实源码，你会发现第 6 节讲的所有容错都在这里汇合：


In [11]:
# ============================================================
# Cell 11：走读 scheduler._review_one_clause（单条款审查）
# ============================================================
import inspect
import review.scheduler as scheduler

print(inspect.getsource(scheduler._review_one_clause))


def _review_one_clause(clause_order: int, clause: dict, contract_meta: dict,
                       rules: list) -> dict:
    """审查单条条款：规则通道优先，否则 LLM + RAG。返回 {verdict, channel}。"""
    content = clause.get("content", "")
    title = clause.get("title") or ""

    # ---- RAG 检索（语义检索 + 可选重排序；不可用自动降级为空）----
    query = f"{clause.get('clause_id')} {title} {content[:200]}".strip()
    try:
        hits = kb.retrieve(query)
        law_context = kb.format_context(hits)
    except Exception as e:
        print(f"[Scheduler] 检索降级: {e}")
        hits, law_context = [], ""

    # ---- 通道一：确定性规则 ----
    rule_verdict = evaluate_clause_rules(content, rules) if content else None
    if rule_verdict:
        # 附上检索到的最相关法条原文，保证结论可追溯
        if hits:
            rule_verdict["law_basis"] = hits[0].page_content[:200]
        return {"verdict": rule_verdict, "channel": "rule"}

    # ---- 通道二：LLM 审查 ----
    model_clause = dict(clause)
    if len(content) > settings.max_clause_chars:
        model_clau

读源码时注意这三处“为什么”：

| 代码位置 | 为什么 |
|---------|--------|
| `kb.retrieve(query)` 包在 try/except 里 | 检索挂了降级为空，审查继续（资料员缺席不影响裁判） |
| 规则命中 `return {...channel: rule}` | **不再调用模型**：红线毫秒级拦截，省一次 LLM 调用 |
| 条款超长 `content[:max_clause_chars]` | 防上下文超限与成本失控，截断处明确标注 |
| `ParseError` → 追加 `retry_note()` 重试（≤2 次） | 模型“偶发话痨”重试即可救回；**provider 级故障则直接 break** |
| 外层 except → `fallback_verdict` | 单条失败绝不拖垮整单 |

### 7.3 整单调度：线程池 + 逐条落库 + 进度

再看外层 `run_review_task_blocking` 怎么把单条款能力变成“整单流水线”：


In [12]:
# ============================================================
# Cell 12：走读 scheduler.run_review_task_blocking（整单调度）
# ============================================================
print(inspect.getsource(scheduler.run_review_task_blocking))


def run_review_task_blocking(task_id: str) -> None:
    """后台执行整单审查（在独立线程中运行，供 asyncio.to_thread 包装）。"""
    task = repo.get_task(task_id)
    if task is None:
        return
    clauses = task.get("clauses") or []
    contract_meta = {
        "contract_name": task.get("contract_name"),
        "party_a": task.get("party_a"),
        "party_b": task.get("party_b"),
        "total_amount": task.get("total_amount"),
    }
    rules = repo.get_enabled_rules()
    repo.set_task_running(task_id)

    try:
        completed = 0
        with cf.ThreadPoolExecutor(max_workers=max(1, settings.workers)) as executor:
            futures = {}
            for order, clause in enumerate(clauses, start=1):
                future = executor.submit(_review_one_clause, order, clause,
                                         contract_meta, rules)
                futures[future] = (order, clause)
            for future in cf.as_completed(futures):
                order, clause = futures[future]
         

**状态机与落库的关键约定**（配合 `db/review_repository.py` 走读）：

1. 任务五态：`pending → running → completed / failed`；
2. 每条结果**独立短事务落库**（`save_result`）：SQLite 默认单写者，用 WAL + 每次新建短连接
   让“8 线程并发写库”与“前端轮询读库”互不阻塞；
3. `current` 字段逐条递增 = 前端进度条“正在审查第 3/30 条”的数据源；
4. 风险汇总（`summary`）在完成时聚合冗余落库，列表页免 GROUP BY；
5. 崩溃遗留的 `pending/running` 任务，启动时统一复位为 `failed`（`mark_interrupted_as_failed`）——
   否则前端永远轮询不到终态（第 33 节会演示）。

引擎的每一块都读完了。下面把它**合体**跑一遍端到端。


---
## Part 8：端到端引擎演示 —— 三通道合体（教学版 = 放慢 + 可控版）
---

### 8.1 为什么要用“假模型”演示？

真实审查要连 vLLM / SiliconFlow，课堂环境未必有。教学版的做法是**注入两个替身**：

| 替身 | 替换谁 | 行为 |
|------|--------|------|
| `FakeKB` | `scheduler.kb` | 检索永远返回空（=“没检索到法条”，走无依据分支） |
| `FakeLLM` | `scheduler.llm_client` | 条款文本不含“故意坏输出”→ 返回合规 JSON（`无`）；含 → 返回一堆非 JSON 废话（制造解析失败） |
| `lambda` | `scheduler.generate_report` | 先关掉报告生成，专心看通道分布（Part 9 再单独开） |

> 这就是“依赖注入”的教学价值：**引擎代码一行不改**，替换掉外部依赖就能离线复现整条链路。

我们设计 5 条条款，让三种通道都出现一次：

| 条款 | 设计 | 期望通道 |
|------|------|---------|
| 第一条 | 违约金上限 30%（超红线） | 规则命中 → `rule` / 高 |
| 第二条 | 知识产权归双方共有 | 规则命中 → `rule` / 中 |
| 第三条 | 管辖地乙方所在地法院 | 规则命中 → `rule` / 中 |
| 第四条 | 普通验收条款（规则放行） | LLM → `llm` / 无 |
| 第五条 | 含“故意坏输出”（模型摆烂） | 重试仍失败 → `fallback` / 低 |


In [13]:
# ============================================================
# Cell 13：端到端 —— 真实 scheduler + 临时 SQLite + Fake LLM/KB
# ============================================================
import uuid

from db import review_repository as repo
from db.database import init_db
from llm.client import LLMResult

init_db()  # 建表 + 首次 seed 3 条出厂规则（写进 SANDBOX/demo.db）

enabled = repo.get_enabled_rules()
print(f"建库并 seed 后，启用的出厂规则 {len(enabled)} 条：")
for r in enabled:
    print(f"    - {r['name']} [{r['matcher_kind']}] 等级={r['risk_level']}")


class FakeKB:
    # 替身知识库：永远检索不到法条（=审查走'未检索到相关法条'分支）
    def retrieve(self, query, k=None):
        return []

    def format_context(self, hits):
        return ""


class FakeLLM:
    # 替身模型：正常条款给合规 JSON；含'故意坏输出'就给非 JSON（制造 ParseError）
    def chat_json(self, messages):
        whole = "\n".join(m["content"] for m in messages)
        if "故意坏输出" in whole:  # 拼接后判断：重试追加的提示词不会稀释触发词
            return LLMResult(text="这不是合法JSON", provider="fake")
        return LLMResult(
            text='{"risk_level": "无", "risk_description": "本条款内容常规，未见风险。", '
                 '"law_basis": "", "suggestion": "无需修改", "confidence": 0.95}',
            provider="fake")


# 注入替身（引擎代码零改动）
import review.scheduler as scheduler
scheduler.kb = FakeKB()
scheduler.llm_client = FakeLLM()
scheduler.generate_report = lambda task_id: None  # 本 Cell 先不生成 docx

demo_clauses = [
    {"clause_id": "第一条", "title": "违约金上限",
     "content": "乙方逾期交付的，违约金上限为合同总金额的30%，且上不封顶。",
     "char_index": 0, "page": 1},
    {"clause_id": "第二条", "title": "知识产权",
     "content": "开发过程中产生的软件著作权归双方共同所有。",
     "char_index": 20, "page": 1},
    {"clause_id": "第三条", "title": "争议解决",
     "content": "协商不成的，任何一方均可向乙方所在地人民法院提起诉讼。",
     "char_index": 40, "page": 1},
    {"clause_id": "第四条", "title": "验收标准",
     "content": "软件功能应符合附件约定，甲方在收到申请后十个工作日内组织验收。",
     "char_index": 60, "page": 1},
    {"clause_id": "第五条", "title": "特殊条款",
     "content": "乙方须满足下列要求，否则视为违约（故意坏输出触发兜底演示）。",
     "char_index": 80, "page": 1},
]

task_id = "task_" + uuid.uuid4().hex[:8]
repo.create_task(task_id, "file_demo", "演示合同.txt", "演示合同",
                 "甲方公司", "乙方公司", "100万元", demo_clauses)
print(f"\n创建审查任务 task_id={task_id}（pending，共 {len(demo_clauses)} 条）...")

scheduler.run_review_task_blocking(task_id)  # 阻塞执行整单（生产环境在 asyncio.to_thread 里跑）

task = repo.get_task(task_id)
print(f"\n最终状态: {task['status']}  |  风险汇总 summary: {task['summary']}")

print("\n逐条结果（channel 分布 = 双通道 + 兜底全部亮相）:")
results = repo.get_results(task_id)
for r in results:
    print(f"  {r['clause_order']}. {r['clause_id']} {r['title']}  -> "
          f"{r['risk_level']}  [{r['channel']}]  conf={r['confidence']}")


建库并 seed 后，启用的出厂规则 3 条：
    - 违约金/赔偿上限超标 [percent_ceiling] 等级=高
    - 知识产权归属不当 [ip_rights] 等级=中
    - 争议管辖地不利 [keyword_any] 等级=中

创建审查任务 task_id=task_78079d74（pending，共 5 条）...

最终状态: completed  |  风险汇总 summary: {'高': 1, '中': 2, '低': 1, '无': 1}

逐条结果（channel 分布 = 双通道 + 兜底全部亮相）:
  1. 第一条 违约金上限  -> 高  [rule]  conf=0.95
  2. 第二条 知识产权  -> 中  [rule]  conf=0.95
  3. 第三条 争议解决  -> 中  [rule]  conf=0.95
  4. 第四条 验收标准  -> 无  [llm]  conf=0.95
  5. 第五条 特殊条款  -> 低  [fallback]  conf=0.0


### 8.2 结果解读：这张表说明了什么？

| 观察点 | 结论 |
|--------|------|
| 第 1~3 条 `channel=rule` | 红线条款根本没进模型——**毫秒级拦截，省 3 次 LLM 调用** |
| 第 4 条 `channel=llm` | 规则放行的普通条款交给模型，模型按提示词给了合规 JSON（`无`） |
| 第 5 条 `channel=fallback` | 模型摆烂 → 重试 2 次仍失败 → 兜底为“低”，**整单仍 completed** |
| `summary={高:1, 中:2, 低:1, 无:1}` | 兜底条记入“低”且带 conf=0，**不会污染高风险统计** |

这就是 PRD“高风险误判率 <1%”红线的实现根基：**高风险只可能来自确定性规则或高置信 LLM 判断，绝不来自‘没审成’**。

再细看规则命中与兜底各一条，感受“可解释性”与“兜底语义”：


In [14]:
# ============================================================
# Cell 14：规则命中 vs 兜底 —— 可解释性长什么样
# ============================================================
rule_rows = [r for r in results if r["channel"] == "rule"]
fb_rows = [r for r in results if r["channel"] == "fallback"]

top = rule_rows[0]
print("◆ 规则通道命中（可解释、可追责）：")
print(f"   {top['clause_id']} {top['title']}")
print(f"   风险描述: {top['risk_description']}")
print(f"   修改建议: {top['suggestion']}")
print(f"   置信度  : {top['confidence']}（确定性规则恒定 0.95，不依赖模型采样）")

fb = fb_rows[0]
print("\n◆ 兜底裁决（模型摆烂时的最后防线）：")
print(f"   {fb['clause_id']} {fb['title']}")
print(f"   风险描述: {fb['risk_description']}")
print(f"   置信度  : {fb['confidence']}（0 = 未完成自动判断，法务必须人工复核）")

print(f"\n任务 risk 汇总: {task['summary']}")
print("注：summary 按每条结果的 risk_level 聚合；fallback 条是'低'，不影响高风险统计。")


◆ 规则通道命中（可解释、可追责）：
   第一条 违约金上限
   风险描述: 【规则:违约金/赔偿上限超标】违约金/赔偿上限可能超过企业红线（合同总额的20%）。
   修改建议: 建议将违约金/赔偿上限调整为企业红线以内，即不超过合同总额的20%。
   置信度  : 0.95（确定性规则恒定 0.95，不依赖模型采样）

◆ 兜底裁决（模型摆烂时的最后防线）：
   第五条 特殊条款
   风险描述: 未能完成自动判断（JSON 解析失败: 未找到 JSON 对象），建议人工复核该条款。
   置信度  : 0.0（0 = 未完成自动判断，法务必须人工复核）

任务 risk 汇总: {'高': 1, '中': 2, '低': 1, '无': 1}
注：summary 按每条结果的 risk_level 聚合；fallback 条是'低'，不影响高风险统计。


---
## Part 9：Word 审查报告生成器
---

### 9.1 报告的“三明治”结构

`review/report_generator.py` 用 python-docx 生成报告，结构是固定的“三明治”：

```
一、基本信息（合同名称/甲乙方/金额/条款数/生成时间）  ← 来自 review_tasks
二、风险概览（高风险红字■ / 中风险黄底■ / …）          ← 来自 summary
三、逐条审查明细（条号+标题 / 原文定位 / 风险描述 / 法条依据 / 修改建议 / 判定通道）
```

### 9.2 为什么颜色要“逐 run 设置”？

python-docx 的字体颜色/高亮**只能设在 run 上，不能设在 paragraph 上**。
所以代码把“一段文字拆成多个 run，各设各的样式”——例如风险概览一行里，
“■ 高风险：1 条”整段红字、中风险加黄底。模块里 `_add_colored_run` 就是为复用这段逻辑而生。

> 也留意颜色语义：红=高、橙=中、绿=低、黑=无；**只有“中”加了黄底高亮**——
> 让法务在 Word 里扫一眼就能抓住“必须处理”的段落。

现在把 Part 8 审完的任务**真正生成一份 docx**，再读回文件验证内容：


In [15]:
# ============================================================
# Cell 15：真实生成 Word 报告并读回验证
# ============================================================
import os

from docx import Document as DocxDocument
from review.report_generator import generate_report

path = generate_report(task_id)  # 基于 SANDBOX/demo.db 里的任务与结果
print(f"报告已生成: {path}")
print(f"文件大小  : {os.path.getsize(path)} 字节")

doc = DocxDocument(path)
paras = [p.text for p in doc.paragraphs if p.text.strip()]
print(f"非空段落数: {len(paras)}")
print("\n挑关键段落预览:")
for line in paras:
    if (line.startswith(("合同智能审查报告", "一、", "二、", "三、")) or "风险等级" in line
            or "判定通道" in line or "风险：1" in line or "风险：2" in line):
        print(f"  · {line}")


报告已生成: C:\Users\13691\AppData\Local\Temp\sm32_6mp1drva\reports\task_78079d74.docx
文件大小  : 37856 字节
非空段落数: 43

挑关键段落预览:
  · 合同智能审查报告
  · 一、基本信息
  · 二、风险概览
  · ■ 高风险：1 条
  · ■ 中风险：2 条
  · ■ 低风险：1 条
  · ■ 无风险：1 条
  · 三、逐条审查明细
  · 风险等级：高
  · 判定通道：确定性规则 · 置信度：0.95
  · 风险等级：中
  · 判定通道：确定性规则 · 置信度：0.95
  · 风险等级：中
  · 判定通道：确定性规则 · 置信度：0.95
  · 风险等级：无
  · 判定通道：大模型审查 · 置信度：0.95
  · 风险等级：低
  · 判定通道：大模型审查 · 置信度：0.00


---
## Part 10：演进回顾 + 课后挑战
---

### 10.1 一图回顾：五个“为什么”

| 设计 | 对应痛点 | 落点 |
|------|---------|------|
| 正则拆条款（而非模型/定长） | 长合同不能整份喂；结构解析要确定 | `contract_splitter.py` |
| 双通道仲裁 | 模型带随机性，红线必须 100% 命中 | `rules.py` + LLM |
| 四段式 Prompt + 裸 JSON 约束 | 模型输出自由文本、还爱编法条 | `context_builder.py` + `verdict_parser.py` |
| 线程池 8 并发 + 逐条落库 | 串行 60s 太慢；单条失败不能拖垮整单 | `scheduler.py` + `db/*` |
| fallback = 低风险·人工复核 | “没审成”不能算高风险、也不能让整单报废 | `verdict_parser.fallback_verdict` |

### 10.2 课后挑战（动手把“为什么”变成“会做”）

1. **加一条新出厂规则**：仿照 `DEFAULT_RULES` 增加 `keyword_any` 规则（如“仲裁机构约定在乙方所在地”），
   重跑 Cell 13 的端到端，观察新条款是否被 `rule` 通道拦截、summary 是否变化。
2. **改拆解器支持“第 X.1 条”**：现实合同里会有“第 10.1 条”这种编号，`_CLAUSE_PATTERN` 目前不识别小数点。
   试改正则并验证 `split_clauses` 输出不串条。
3. **断网试验兜底**：把 `FakeLLM` 换成“直接抛 RuntimeError”的模型，观察单条条款最后落在哪个通道、
   整单状态是否仍为 `completed`——并解释为什么 fallback 条不会进高风险统计。
4. **读一读 `llm/client.py` 的模块头**：回答“为什么 provider 层重试=0、故障在 provider 之间切换”，
   想清楚这和调度器内层重试的分工边界。

### 10.3 下一节预告

引擎在命令行/notebook 里能审合同了。但法务要的不是 notebook——第 33 节做**服务化接线**：
用 FastAPI 把引擎包成“传文件→拿报告”的 HTTP 服务，接上 Vue3 前端与 Docker 部署，
再做一次完整的**端到端验收 Demo**（上传 → 轮询进度 → 三类结果核对 → 报告下载）。

> 本节完 —— 记得把你觉得最妙的“为什么”讲给同桌听。
